In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 66.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from gensim.models import Word2Vec
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Load the dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"])
df_val = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"])

df_train

df_train = df_train[df_train['lang'].isin(['ar', 'ko', 'te'])]
df_val = df_val[df_val['lang'].isin(['ar', 'ko', 'te'])]


df_train.head()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


,question,context,lang,answerable,answer_start,answer,answer_inlang
4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,None
4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,None
4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,None
4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),None
4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,None


In [ ]:
def cleanDf(df):
    pattern = re.compile(r"[?؟,;\/\\\[\]#():]")
    # pattern_context = re.compile(r"[?؟,;\/\\\[\]#():.]")
    df['question'] = df['question'].apply(lambda x: pattern.sub("", x))
    df['context'] = df['context'].apply(lambda x: pattern.sub("", x))
    return df

def word_overlap(context, question):
    context_words = set(context.lower().split())
    question_words = set(question.lower().split())

    overlap = context_words.intersection(question_words)

    if len(question_words) == 0:
        return 0.0
    return len(overlap) / len(question_words)

def train_word2vec(df, embedding_dim=100):
    sentences = []
    for _, row in df.iterrows():
        sentences.append(row['context'].lower().split())
        sentences.append(row['question'].lower().split())

    model = Word2Vec(sentences, vector_size=100, window=10, min_count=1, sg=1, epochs=30)

    return model

def avg_embedding(tokens, wv, dim=100):
    vecs = [wv[t] for t in tokens if t in wv]
    if len(vecs) == 0:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)


def cosine_similarity(vec1, vec2):
    if np.linalg.norm(vec1) == 0 or np.linalg.norm(vec2) == 0:
        return 0.0
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def build_features(df, embedder):
    q_emb = embedder.encode(df['question'].tolist(), convert_to_tensor=True)
    c_emb = embedder.encode(df['context'].tolist(), convert_to_tensor=True)
    cos_sim = torch.nn.functional.cosine_similarity(q_emb, c_emb).unsqueeze(1)
    X = torch.cat([q_emb, c_emb, cos_sim], dim=1)
    return X

def encode_sentences(df, embedder):
    q_emb = embedder.encode(df["question"].tolist(), convert_to_tensor=True, device=device)
    c_emb = embedder.encode(df["context"].tolist(), convert_to_tensor=True, device=device)
    return q_emb, c_emb


## Simple BaseLine logistic regression

In [ ]:
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self,x):
        return self.linear(x)

In [ ]:
df_train = cleanDf(df_train)
df_val = cleanDf(df_val)

languages = ['ar', 'ko', 'te']

results = {}


trained_models = {}
trained_models2 = {}

embedding_dim = 100

def build_text(question, context):
    return "[CLS]" + question + "[SEP]" + context + "[SEP]"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

embedder = SentenceTransformer("intfloat/multilingual-e5-base").to(device)

for lang in languages:
    print(f"Training and evaluating for language: {lang.upper()}")

    df_train_lang = df_train[df_train['lang'] == lang]
    df_val_lang = df_val[df_val['lang'] == lang]

    X_train_tensor = build_features(df_train_lang, embedder).to(device)
    X_val_tensor = build_features(df_val_lang, embedder).to(device)

    y_train_tensor = torch.tensor(df_train_lang['answerable'].values, dtype=torch.float32, device=device).view(-1, 1)
    y_val_tensor = torch.tensor(df_val_lang['answerable'].values, dtype=torch.float32, device=device).view(-1, 1)

    input_dim = X_train_tensor.shape[1]
    model = LogisticRegressionModel(input_dim=input_dim).to(device)
    model2 = LogisticRegressionModel(input_dim=input_dim).to(device)

    num_positive = (y_train_tensor == 1).sum().item()
    num_negative = (y_train_tensor == 0).sum().item()

    # Weights
    pos_weight = torch.tensor(num_negative / num_positive, dtype=torch.float32, device = device)


    criterion = nn.BCEWithLogitsLoss()
    criterion2 = nn.BCEWithLogitsLoss(pos_weight = pos_weight)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.01)

    num_epochs = 100


    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()

        # Weighted model
        optimizer2.zero_grad()
        outputs2 = model2(X_train_tensor)
        loss2 = criterion2(outputs2, y_train_tensor)
        loss2.backward()
        optimizer2.step()

        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Regular Loss: {loss.item():.4f}")
            print(f"Epoch {epoch+1}/{num_epochs}, Weighted Loss: {loss2.item():.4f}")

    trained_models[f"model_{lang}"] = model
    trained_models2[f"model_{lang}"] = model2

    print(f"Stored model_{lang} in memory")


    with torch.no_grad():
        logits = model(X_val_tensor)
        logits2 = model2(X_val_tensor)


        y_pred_class = (torch.sigmoid(logits) >= 0.5).float()
        y_pred_class2 = (torch.sigmoid(logits2) >= 0.5).float()

        y_true = y_val_tensor.cpu().numpy()

        y_pred_np = y_pred_class.cpu().numpy()
        y_pred_np2 = y_pred_class2.cpu().numpy()

        report = classification_report(y_true, y_pred_np, target_names=['Impossible', 'Answerable'])
        report2 = classification_report(y_true, y_pred_np2, target_names=['Impossible', 'Answerable'])

        print(f"Regular Classification Report for {lang.upper()}:\n{report}")
        print(f"Weightet Classification Report for {lang.upper()}:\n{report2}")




Using device: cuda


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 10/100, Regular Loss: 0.1250
Epoch 10/100, Weighted Loss: 0.0300
Epoch 20/100, Regular Loss: 0.1297
Epoch 20/100, Weighted Loss: 0.0251
Epoch 30/100, Regular Loss: 0.1331
Epoch 30/100, Weighted Loss: 0.0215
Epoch 40/100, Regular Loss: 0.1264
Epoch 40/100, Weighted Loss: 0.0187
Epoch 50/100, Regular Loss: 0.1183
Epoch 50/100, Weighted Loss: 0.0165
Epoch 60/100, Regular Loss: 0.1140
Epoch 60/100, Weighted Loss: 0.0147
Epoch 70/100, Regular Loss: 0.1124
Epoch 70/100, Weighted Loss: 0.0133
Epoch 80/100, Regular Loss: 0.1105
Epoch 80/100, Weighted Loss: 0.0121
Epoch 90/100, Regular Loss: 0.1085
Epoch 90/100, Weighted Loss: 0.0111
Epoch 100/100, Regular Loss: 0.1066
Epoch 100/100, Weighted Loss: 0.0102
Stored model_ko in memory
Regular Classification Report for KO:
              precision    recall  f1-score   support

  Impossible       0.00      0.00      0.00        19
  Answerable       0.95      1.00      0.97       337

    accuracy                           0.95       356
   mac

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 10/100, Regular Loss: 0.1468
Epoch 10/100, Weighted Loss: 0.0389
Epoch 20/100, Regular Loss: 0.1603
Epoch 20/100, Weighted Loss: 0.0325
Epoch 30/100, Regular Loss: 0.1597
Epoch 30/100, Weighted Loss: 0.0276
Epoch 40/100, Regular Loss: 0.1472
Epoch 40/100, Weighted Loss: 0.0237
Epoch 50/100, Regular Loss: 0.1386
Epoch 50/100, Weighted Loss: 0.0206
Epoch 60/100, Regular Loss: 0.1366
Epoch 60/100, Weighted Loss: 0.0181
Epoch 70/100, Regular Loss: 0.1339
Epoch 70/100, Weighted Loss: 0.0160
Epoch 80/100, Regular Loss: 0.1312
Epoch 80/100, Weighted Loss: 0.0143
Epoch 90/100, Regular Loss: 0.1287
Epoch 90/100, Weighted Loss: 0.0129
Epoch 100/100, Regular Loss: 0.1262
Epoch 100/100, Weighted Loss: 0.0117
Stored model_te in memory
Regular Classification Report for TE:
              precision    recall  f1-score   support

  Impossible       0.00      0.00      0.00        93
  Answerable       0.76      1.00      0.86       291

    accuracy                           0.76       384
   mac

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import classification_report
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

languages = ['ar', 'ko', 'te']
model_name = "distilbert-base-multilingual-cased"

models = {lang: AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device) for lang in languages}
tokenizer = AutoTokenizer.from_pretrained(model_name)

def encode_examples(df, tokenizer, max_length=256):
    inputs = tokenizer(
        list(df["question"]),
        list(df["context"]),
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )
    labels = torch.tensor(df["answerable"].values, dtype=torch.long)
    return inputs, labels

for lang in languages:
    print(f"\nTraining model for language: {lang.upper()}")

    df_train_lang = df_train[df_train['lang'] == lang]
    df_val_lang = df_val[df_val['lang'] == lang]
    train_inputs, train_labels = encode_examples(df_train_lang, tokenizer)
    train_dataset = TensorDataset(train_inputs['input_ids'], train_inputs['attention_mask'], train_labels)

    labels_np = train_labels.numpy()
    class_counts = np.bincount(labels_np)
    class_weights = 1. / class_counts
    samples_weight = np.array([class_weights[label] for label in labels_np])
    samples_weight = torch.from_numpy(samples_weight).float()
    sampler = WeightedRandomSampler(samples_weight, num_samples=len(samples_weight), replacement=True)


    train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler)

    val_inputs, val_labels = encode_examples(df_val_lang, tokenizer)
    val_input_ids = val_inputs['input_ids'].to(device)
    val_attention_mask = val_inputs['attention_mask'].to(device)

    model = models[lang]
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss()

    model.train()
    num_epochs = 5
    for epoch in range(num_epochs):
        running_loss = 0.0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [{lang.upper()}]", leave=False)
        for step, batch in enumerate(loop, start=1):
            input_ids, attention_mask, y = [b.to(device) for b in batch]
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        avg_loss = running_loss / len(train_loader)
        print(f"Epoch {epoch+1} completed, Average Loss: {avg_loss:.4f}")

    model.eval()
    with torch.no_grad():
        outputs = model(input_ids=val_input_ids, attention_mask=val_attention_mask)
        y_pred = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        y_true = val_labels.numpy()

    print(f"Classification Report for {lang.upper()}:\n")
    print(classification_report(y_true, y_pred, target_names=['Impossible', 'Answerable']))


In [ ]:
test_dir = "/content/drive/MyDrive/translated_data"

test_path = os.path.join(test_dir, "test.json")
df_test = pd.read_json(test_path)

X_test_tensor = build_features(df_test, embedder).to(device)
y_test_tensor = torch.tensor(df_test['answerable'].values, dtype=torch.float32, device=device).view(-1, 1)


model_to_use = trained_models2['model_ko']

X_test_tensor = build_features(df_test, embedder).to(device)
with torch.no_grad():
    logits = model_to_use(X_test_tensor)
    y_pred_class = (torch.sigmoid(logits) >= 0.5).float()

    y_true = y_test_tensor.cpu().numpy()
    y_pred = y_pred_class.cpu().numpy()

    print(classification_report(
        y_true,
        y_pred,
        target_names=['Impossible', 'Answerable'],
        digits=3
    ))


              precision    recall  f1-score   support

  Impossible      0.500     0.250     0.333         4
  Answerable      0.833     0.938     0.882        16

    accuracy                          0.800        20
   macro avg      0.667     0.594     0.608        20
weighted avg      0.767     0.800     0.773        20

